In [16]:
import pandas as pd
import glob
from inference import get_model_available, get_data_academic_risk, predict_academic_risk
import numpy as np
from complete_data_to_model import complete_data_to_model, limpieza_df_con_gpa_socioeconomico_interes

In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
import os

os.add_dll_directory('C:\\Program Files\\IBM\\SQLLIB\\BIN')
# conectar a la base de datos IBM Db2
import ibm_db

In [19]:
conn_str = 'DATABASE=' + os.getenv('DATABASE') + ';HOSTNAME=' + os.getenv('HOSTNAME') + ';PORT=' + os.getenv('PORT') + ';PROTOCOL=TCPIP;UID=' + os.getenv('USERNAME_DB') + ';PWD=' + os.getenv('PASSWORD_DB') + ';'
conn = ibm_db.connect(conn_str, '', '')
# conn = ibm_db.connect(os.getenv('DATABASE'), os.getenv('USERNAME_DB'), os.getenv('PASSWORD_DB')) # os.getenv('HOSTNAME'), os.getenv('PORT')

if conn:
    print("Conexión exitosa")
else:
    print("Error al conectar")

Conexión exitosa


In [20]:
anio_base, termino_base = 2026, 1  # Periodo actual
cod_materia_objetivo = "ESTT3002" #"CCPG1042"# "MATG1058", 'CCPG1052'  # Materia objetivo

In [21]:
def get_data_from_query(sql_str):
    stmt_select  = ibm_db.exec_immediate(conn, sql_str)
    
    # Fetch all rows
    data_list = []
    result = ibm_db.fetch_assoc(stmt_select)
    while result:
        # print(result)
        data_list.append(result)
        result = ibm_db.fetch_assoc(stmt_select)
    return data_list

In [22]:
modelo, label_encoders, feature_info = get_model_available()


📂 Cargando modelos y configuración...

✅ Modelo seleccionado: Random Forest: ../models/random_forest_model.pkl
✓ Modelo cargado desde '../models/random_forest_model.pkl'


In [23]:
def make_analysis(list_matricula, cod_materia):
    mi_df, list_student_off = get_data_academic_risk(list_matricula, cod_materia, label_encoders)
    if mi_df.empty:
        print("⚠️ No se encontraron datos para los estudiantes proporcionados. Asegúrate de que las matrículas sean correctas y que los estudiantes hayan tomado la materia objetivo.")
        return {}, mi_df, {}, list_student_off

    print("Debe coinicidir la cantidad:",mi_df.shape[0] == len(list_matricula))
    if mi_df.shape[0] != len(list_matricula):
        print("⚠️ Advertencia: Algunos estudiantes no tienen datos completos para el análisis.")

    results = predict_academic_risk(modelo, feature_info, mi_df)

    stats = results['statistics']
    # print(f"📊 {stats['pred_aprobar']} estudiantes aprobarán ({stats['pct_aprobar']:.2f}%)")
    
    return results, mi_df, stats, list_student_off

In [24]:
dic_terms = {
    "100II": 1,
    "200I": 2,
    "200II": 3,
    "300I": 4,
    "300II": 5,
    "400I": 6,
    "400II": 7,
    "500I": 8,
    "500II": 9
}
#min. terminos registrados de estudiantes que no hayan tomado la materia objetivo

### Datos de historico completo desde 2020 a 2025 2S

In [25]:
list_names_files = glob.glob("../data/riesgo_academico/all_*")
df_materias = pd.read_csv("../data/riesgo_academico/dificultad_materia.csv")

In [26]:
df_complete = pd.DataFrame()

for file in list_names_files:
    if not "2026_2S" in file:
        print("file", file)
        split_file = file.split("_")
        termino = split_file[-1].split(".")[0]
        if termino == "3S":
            continue
        
        df = pd.read_csv(file)

        df["anio"] = split_file[-2]
        df["termino"] = termino
        if not("MATERIA" in df.keys()):
            df = pd.merge(df, df_materias[["CODIGOMATERIA", "MATERIA"]], left_on="COD_MATERIA_ACAD_MO", right_on="CODIGOMATERIA")

        df_complete = pd.concat([df_complete, df], ignore_index=True)

file ../data/riesgo_academico\all_2020_1S.csv


file ../data/riesgo_academico\all_2020_2S.csv
file ../data/riesgo_academico\all_2021_1S.csv
file ../data/riesgo_academico\all_2021_2S.csv
file ../data/riesgo_academico\all_2022_1S.csv
file ../data/riesgo_academico\all_2022_2S.csv
file ../data/riesgo_academico\all_2023_1S.csv
file ../data/riesgo_academico\all_2023_2S.csv
file ../data/riesgo_academico\all_2024_1S.csv
file ../data/riesgo_academico\all_2024_2S.csv
file ../data/riesgo_academico\all_2025_1S.csv
file ../data/riesgo_academico\all_2025_2S.csv
file ../data/riesgo_academico\all_2026_1S.csv


In [27]:
df_complete["COD_MATERIA_ACAD_MO"].nunique()

707

In [28]:
df_complete["COD_ESTUDIANTE"] = df_complete["COD_ESTUDIANTE"].astype(str)

In [29]:
df_complete.shape

(448261, 30)

In [30]:
df_complete

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,FACIL,MODERADA,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL
0,201160178,ACUG1035,AP,1,87,93,"9,00","7,90",53.0,59.0,...,0,2,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
1,201310353,ACUG1035,AP,1,85,89,"8,70","7,90",59.0,61.0,...,0,2,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
2,201313869,ACUG1035,AP,1,86,89,"8,75","7,90",59.0,56.0,...,1,1,0,1,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
3,201507649,ACUG1035,AP,1,83,93,"8,80","7,90",49.0,65.0,...,0,2,2,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
4,201607884,ACUG1035,AP,1,95,95,"9,50","7,90",39.0,72.0,...,1,1,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448256,202508370,TURG2037,AP,1,0,0,0.0,7.22,5.0,84.0,...,0,0,1,4,NaN,TURG2022,DERECHO TURÍSTICO Y AMBIENTAL,2026,1S,NaN
448257,202508669,TURG2037,AP,1,0,0,0.0,7.22,5.0,74.0,...,0,0,1,4,NaN,TURG2022,DERECHO TURÍSTICO Y AMBIENTAL,2026,1S,NaN
448258,202509493,TURG2037,AP,1,0,0,0.0,7.22,5.0,84.0,...,0,0,1,5,NaN,TURG2022,DERECHO TURÍSTICO Y AMBIENTAL,2026,1S,NaN
448259,202509634,TURG2037,AP,1,0,0,0.0,7.22,5.0,77.0,...,0,0,1,5,NaN,TURG2022,DERECHO TURÍSTICO Y AMBIENTAL,2026,1S,NaN


In [31]:
# group by anio and termino
df_complete.groupby(["anio", "termino"]).size() 

anio  termino
2020  1S         35871
      2S         34086
2021  1S         32727
      2S         31286
2022  1S         29507
      2S         28859
2023  1S         28301
      2S         28195
2024  1S         28707
      2S         28488
2025  1S         30264
      2S         30609
2026  1S         81361
dtype: int64

#### Carrera con datos completos de estudiantes

In [17]:
df_carreras_estudiantes = pd.read_csv("../data/riesgo_academico/carreras_estudiantes.csv")

In [18]:
df_carreras_estudiantes["CODESTUDIANTE"] = df_carreras_estudiantes["CODESTUDIANTE"].astype(str)
df_carreras_estudiantes["CODESTUDIANTE"] = df_carreras_estudiantes["CODESTUDIANTE"].str.strip()

In [19]:
# # agrupa los CODESTUDIANTE y CARRERA y cuenta la cantidad de ocurrencias. Con CODESTUDIANTE y CARRERA como columnas separadas
df_carreras_estudiantes_new = df_carreras_estudiantes.groupby(['CODESTUDIANTE', 'CARRERA', 'ANIO']).size().reset_index(name='CANTIDAD').copy()

In [20]:
# Obtener el año máximo por cada combinación de CODESTUDIANTE y CARRERA
df_carreras_estudiantes_max_anio = df_carreras_estudiantes_new.loc[
    df_carreras_estudiantes_new.groupby(['CODESTUDIANTE', 'CARRERA'])['ANIO'].idxmax()
][['CODESTUDIANTE', 'CARRERA', 'ANIO']].reset_index(drop=True)

In [21]:
# Conservar solo una fila por CODESTUDIANTE con el ANIO máximo
df_carreras_estudiantes_max_anio = (
    df_carreras_estudiantes_max_anio
    .sort_values('ANIO', ascending=False)
    .groupby('CODESTUDIANTE', as_index=False)
    .first()
)

In [22]:
df_carreras_estudiantes_max_anio

,CODESTUDIANTE,CARRERA,ANIO
0,198802423,Acuicultura,2020
1,198901423,Electricidad,2020
2,198903122,Arqueología,2024
3,198905267,Acuicultura,2020
4,199002130,Electricidad,2025
...,...,...,...
19694,202590329,Movilidad Nacional,2025
19695,202590337,Movilidad Nacional,2025
19696,202590345,Movilidad Nacional,2025
19697,202590352,Movilidad Nacional,2025


In [23]:
df_carreras_estudiantes_max_anio["CODESTUDIANTE"].value_counts()

CODESTUDIANTE
202590360    1
198802423    1
198901423    1
202590204    1
202590196    1
            ..
199500075    1
199202169    1
199201526    1
199002130    1
198905267    1
Name: count, Length: 19699, dtype: int64

In [24]:
df_carreras_estudiantes_max_anio[df_carreras_estudiantes_max_anio["CODESTUDIANTE"] == "201908803"]

,CODESTUDIANTE,CARRERA,ANIO
8433,201908803,Alimentos,2025


In [25]:
df_complete.shape, df_carreras_estudiantes_max_anio.shape

((366900, 30), (19699, 3))

In [26]:
# agregar a df_complete CARRERA desde df_carreras_estudiantes haciendo merge con COD_ESTUDIANTE de df_complete y CODESTUDIANTE de df_carreras_estudiantes
df_complete = pd.merge(df_complete, df_carreras_estudiantes_max_anio[['CODESTUDIANTE', 'CARRERA']], left_on='COD_ESTUDIANTE', right_on='CODESTUDIANTE', how='inner')
df_complete = df_complete.drop(columns=['CODESTUDIANTE'])

In [27]:
df_complete["termino_num"] = df_complete["termino"].map({"1S": 1, "2S": 2, "3S": 3})
df_complete['DIFICULTAD_MO'] = df_complete['DIFICULTAD_MO'].str.replace(',', '.').astype(float)
df_complete['PROM_MAT_REPROBADAS1'] = df_complete['PROM_MAT_REPROBADAS1'].str.replace(',', '.').astype(float)


In [28]:
df_complete

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num
0,201160178,ACUG1035,AP,1,87,93,"9,00",7.90,53.0,59.0,...,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
1,201310353,ACUG1035,AP,1,85,89,"8,70",7.90,59.0,61.0,...,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
2,201313869,ACUG1035,AP,1,86,89,"8,75",7.90,59.0,56.0,...,0,1,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
3,201507649,ACUG1035,AP,1,83,93,"8,80",7.90,49.0,65.0,...,2,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
4,201607884,ACUG1035,AP,1,95,95,"9,50",7.90,39.0,72.0,...,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366895,202300422,TURG2037,AC,1,0,0,"0,00",7.22,27.0,73.0,...,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,92",Turismo,2
366896,202104683,TURG2037,AC,1,0,0,"0,00",7.22,37.0,65.0,...,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,54",Turismo,2
366897,202111548,TURG2037,AC,1,0,0,"0,00",7.22,36.0,66.0,...,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,53",Turismo,2
366898,202103719,TURG2037,AC,1,0,0,"0,00",7.22,33.0,67.0,...,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,25",Turismo,2


#### GPA

In [29]:
list_gpa_gemeral = glob.glob("../data/riesgo_academico/gpa_*S.csv")

In [30]:
df_gpa_general = pd.DataFrame()
for file_gpa_gen in list_gpa_gemeral:
    print(file_gpa_gen)
    df_tmp = pd.read_csv(file_gpa_gen)
    df_gpa_general = pd.concat([df_gpa_general, df_tmp], ignore_index=True)

../data/riesgo_academico\gpa_general_2020_0S.csv
../data/riesgo_academico\gpa_general_2020_1S.csv
../data/riesgo_academico\gpa_general_2020_2S.csv
../data/riesgo_academico\gpa_general_2021_1S.csv
../data/riesgo_academico\gpa_general_2021_2S.csv
../data/riesgo_academico\gpa_general_2022_1S.csv
../data/riesgo_academico\gpa_general_2022_2S.csv
../data/riesgo_academico\gpa_general_2023_1S.csv
../data/riesgo_academico\gpa_general_2023_2S.csv
../data/riesgo_academico\gpa_general_2024_1S.csv
../data/riesgo_academico\gpa_general_2024_2S.csv
../data/riesgo_academico\gpa_general_2025_1S.csv
../data/riesgo_academico\gpa_general_2025_2S.csv


In [31]:
df_gpa_general['COD_ESTUDIANTE'] = df_gpa_general['COD_ESTUDIANTE'].astype(str).str.strip()

In [32]:
df_gpa_general = df_gpa_general.drop_duplicates()

In [33]:
df_gpa_general

,COD_ESTUDIANTE,ANIO,TERMINO,DENOMINADOR,NUMERADOR
0,200818094,2020,2S,4.0,32.4
1,200110245,2020,2S,6.0,36.6
2,202010070,2020,2S,6.0,46.5
3,202010120,2020,2S,6.0,40.5
4,202010625,2020,2S,6.0,39.0
...,...,...,...,...,...
99823,202414041,2025,2S,NaN,NaN
99824,202501375,2025,2S,NaN,NaN
99825,202505467,2025,2S,NaN,NaN
99826,202509501,2025,2S,NaN,NaN


#### Socioeconomico del estudiante objetivo

In [34]:
df_socioeconomico = pd.read_csv("../data/riesgo_academico/socioeconomico_17944.csv")
df_socioeconomico['CODESTUDIANTE'] = df_socioeconomico['CODESTUDIANTE'].astype(str).str.strip()

C:\Users\saraujo\AppData\Local\Temp\ipykernel_31160\1392310176.py:1: DtypeWarning: Columns (10,41,42,66,123) have mixed types. Specify dtype option on import or set low_memory=False.
  df_socioeconomico = pd.read_csv("../data/riesgo_academico/socioeconomico_17944.csv")


In [35]:
df_socioeconomico["GASTOS_RUBRO"] = ((df_socioeconomico["ALIMENTACION"] + df_socioeconomico["TRANSPORTE"] + df_socioeconomico["SERVICIOS"] +
  df_socioeconomico['ARRIENDO'] + df_socioeconomico['ALICUOTAS'] + df_socioeconomico['VESTIMENTA'] + df_socioeconomico['SALUD'] + df_socioeconomico['EDUCACION'] +
  df_socioeconomico['TARJETACREDITO'] + df_socioeconomico['ENTRETENIMIENTO'] + df_socioeconomico['OTROS']) / df_socioeconomico["NUMEROSFAMILIARES"]).round(2)


In [36]:
# eliminar duplicados de CODESTUDIANTE dejando la primera ocurrencia
df_socioeconomico = df_socioeconomico.drop_duplicates(subset=['CODESTUDIANTE'], keep='first')

### Estudiantes que estan viendo actualmente la materia objetivo

In [37]:
dir_estudiantes_viendo = '../database/planificacion_aprobacion/estudiantes_actualmente_viendo.sql'


In [38]:
with open(dir_estudiantes_viendo, 'r', encoding='utf-8') as file:
    sql_estudiantes_viendo_base = file.read()

In [39]:
sql_estudiantes_viendo = sql_estudiantes_viendo_base.split("------------")[0]
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('\n', ' ')
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('TT', f"{termino_base}S")
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('AAAA', f"{anio_base}")
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('xxxxx', f"{cod_materia_objetivo}")

In [40]:
sql_estudiantes_viendo

"SELECT COD_ESTUDIANTE, COD_MATERIA_ACAD, tm.NOMBRE, tpa.nombre carrera, hd.*  FROM espol.HISTORIA_ANIO  hd  INNER JOIN espol.TBL_MATERIA tm ON hd.COD_MATERIA_ACAD =tm.CODIGOMATERIA  INNER JOIN espol.TBL_PROGRAMA_ACADEMICO tpa ON tpa.CODCARRERA =hd.COD_CARRERA and tpa.CODDIVISION =hd.COD_DIVISION AND tpa.CODESPECIALIZ =hd.COD_ESPECIALIZ WHERE tm.codigomateria IN ('ESTT3002') AND termino='2S' AND ANIO='2025' AND hd.ESTADO_MAT_TOMADA <>'AN' ;  "

In [41]:
# Fetch all rows
data_list = get_data_from_query(sql_estudiantes_viendo)

In [42]:
data_list

[]

In [43]:
if len(data_list) == 0:
    print("⚠️ No se encontraron estudiantes actualmente viendo la materia objetivo", cod_materia_objetivo)
    print("+"*100)
    # stop run
    # exit()

⚠️ No se encontraron estudiantes actualmente viendo la materia objetivo ESTT3002
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


In [44]:
if len(data_list) == 0:
    df_estudiantes_viendo = pd.DataFrame(columns=["COD_ESTUDIANTE", "NOMBRE", "CARRERA", "COD_MATERIA_ACAD", "ANIO", "TERMINO", "VEZ_TOMADA"])
else:
    df_estudiantes_viendo = pd.DataFrame(data_list)[["COD_ESTUDIANTE", "NOMBRE", "CARRERA", "COD_MATERIA_ACAD", "ANIO", "TERMINO", "VEZ_TOMADA"]].copy()

In [45]:
# hacer antes el copy porque si no no sabe si es en el copy o en el original
df_estudiantes_viendo["COD_MATERIA_ACAD"] = df_estudiantes_viendo["COD_MATERIA_ACAD"].astype(str)
df_estudiantes_viendo["COD_MATERIA_ACAD"] = df_estudiantes_viendo["COD_MATERIA_ACAD"].str.strip() 

In [46]:
df_estudiantes_viendo.shape

(0, 7)

In [47]:
df_estudiantes_viendo.head(8)

,COD_ESTUDIANTE,NOMBRE,CARRERA,COD_MATERIA_ACAD,ANIO,TERMINO,VEZ_TOMADA


In [48]:
df_estudiantes_viendo["CARRERA"].value_counts()

Series([], Name: count, dtype: int64)

### Prerequisitos necesarios de la materia objetivo

In [49]:
dir_pre_co_requisitos = '../database/planificacion_aprobacion\\corequisito_prerequisito_materias.sql'

In [50]:
# queries SQL parameters
with open(dir_pre_co_requisitos, 'r', encoding='utf-8') as file:
    sql_pre_co_requisitos_base = file.read()


In [51]:
sql_pre_co_requisitos = sql_pre_co_requisitos_base.split("------------")[0]
sql_pre_co_requisitos = sql_pre_co_requisitos.replace('\n', ' ')
sql_pre_co_requisitos = sql_pre_co_requisitos.replace('xxxxx', f"{cod_materia_objetivo}")

In [52]:
sql_pre_co_requisitos

"SELECT distinct pa.nombre carrera, m.codigomateria, m.nombre materia , mm.nivel, m.HORASDOCENCIASEM,m.HORASPRACTICASSEM, m.HORASAUTONOMSEM, m.nombreingles , CASE when m.tipomateria = 'T' then 'teórica' when m.tipomateria = 'I' then 'importada' when m.tipomateria = 'p' then 'practica' when m.tipomateria = 'r' then 'teóricoPractica' when m.tipomateria = 'n' then 'nivelcero' when m.tipomateria = 'l' then 'con laboratorio' when m.tipomateria = 'a' then 'laboratorio' when m.tipomateria = 'g' then 'graduacion' when m.tipomateria = 's' then 'teórico práctico sin laboratorio' when m.tipomateria = 'm' then 'modular' when m.tipomateria = 'c' then 'mención' when m.tipomateria = 'z' then 'paralelo practico unido a teorico x codigomateria'  when m.tipomateria = 'o' then 'integradora' when m.tipomateria = 'v' then 'investigacion' else m.tipomateria end tipomateria, tc.nombre tipocredito , case when m.clasifmateria = 'g' then 'general' when m.clasifmateria = 'c' then 'complementaria' when m.clasifma

In [53]:
# Fetch all rows
data_list = get_data_from_query(sql_pre_co_requisitos)

In [54]:
cod_materia_objetivo

'ESTT3002'

In [55]:
data_list

[{'CARRERA': 'Tecnología Superior en Mecatrónica',
  'CODIGOMATERIA': None,
  'MATERIA': 'METROLOGÍA',
  'NIVEL': '100I',
  'HORASDOCENCIASEM': 4,
  'HORASPRACTICASSEM': 2,
  'HORASAUTONOMSEM': 6,
  'NOMBREINGLES': 'METROLOGY',
  'TIPOMATERIA': 'teóricoPractica',
  'TIPOCREDITO': 'FORMACIÓN BÁSICA',
  'CLASIFICACION': 'general',
  'IDMATERIAREQ': None,
  'MATERIA_REQUISITO': None,
  'TIPO': None,
  'NOMBREUNIDAD': 'Facultad de Ingeniería en Mecánica y Ciencias de la Producción',
  'HORASDOCENCIASEMESTRE': 64,
  'HORASPRACTICASEMESTRE': 32,
  'HORASAUTONOMASSEMESTRE': 96}]

In [56]:
df_pre_requisito = pd.DataFrame(data_list)[["CARRERA", "NIVEL", "MATERIA_REQUISITO", "CODIGOMATERIA", "TIPO", "TIPOMATERIA", "MATERIA"]].copy()

In [57]:
df_pre_requisito["NUM_MIN_TERMINOS"] = df_pre_requisito["NIVEL"].map(dic_terms)

In [58]:
df_pre_requisito

,CARRERA,NIVEL,MATERIA_REQUISITO,CODIGOMATERIA,TIPO,TIPOMATERIA,MATERIA,NUM_MIN_TERMINOS
0,Tecnología Superior en Mecatrónica,100I,None,None,None,teóricoPractica,METROLOGÍA,NaN


In [59]:
try:
    materia_objetivo = df_pre_requisito["MATERIA"].iloc[0]
    print("Materia objetivo:", materia_objetivo, "con codigo", cod_materia_objetivo)
except IndexError:
    print("⚠️ No se encontraron prerrequisitos para la materia objetivo. Asumiendo que no tiene prerrequisitos.")
    materia_objetivo = "No tenemos en el nombre desde las prerequisitos, porque no tiene prerequisitos"

Materia objetivo: METROLOGÍA con codigo ESTT3002


In [60]:
df_pre_requisito_cp = df_pre_requisito.copy()
df_pre_requisito_cp

,CARRERA,NIVEL,MATERIA_REQUISITO,CODIGOMATERIA,TIPO,TIPOMATERIA,MATERIA,NUM_MIN_TERMINOS
0,Tecnología Superior en Mecatrónica,100I,None,None,None,teóricoPractica,METROLOGÍA,NaN


In [61]:
df_pre_requisito = df_pre_requisito[df_pre_requisito["TIPO"] == "PR"].copy()

In [62]:
if df_pre_requisito.shape[0] == 0:
    print("⚠️ No se encontraron prerrequisitos para la materia", cod_materia_objetivo)
    print("Si no hay prerequisito, quiere decir que es de primera linea(por ahora). No se toma en cuenta.")

⚠️ No se encontraron prerrequisitos para la materia ESTT3002
Si no hay prerequisito, quiere decir que es de primera linea(por ahora). No se toma en cuenta.


In [63]:
df_pre_requisito.shape

(0, 8)

In [64]:
df_pre_requisito

,CARRERA,NIVEL,MATERIA_REQUISITO,CODIGOMATERIA,TIPO,TIPOMATERIA,MATERIA,NUM_MIN_TERMINOS


### Estudiantes de prerequisitos, cuales han aprobado para considerar en la planificacion

In [65]:
lis_carreras_pre = df_pre_requisito["CARRERA"].unique().tolist()

In [66]:
lis_carreras_pre

[]

In [67]:
lis_cod_materias_pre = [i.strip() for i in df_pre_requisito["CODIGOMATERIA"].unique().tolist()]

In [68]:
lis_cod_materias_pre

[]

In [69]:
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_base.split("------------")[0]
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('\n', ' ')
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('TT', f"{termino_base}S")
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('AAAA', f"{anio_base}")
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('xxxxx', "','".join(lis_cod_materias_pre))

In [70]:
sql_estudiantes_viendo_pre

"SELECT COD_ESTUDIANTE, COD_MATERIA_ACAD, tm.NOMBRE, tpa.nombre carrera, hd.*  FROM espol.HISTORIA_ANIO  hd  INNER JOIN espol.TBL_MATERIA tm ON hd.COD_MATERIA_ACAD =tm.CODIGOMATERIA  INNER JOIN espol.TBL_PROGRAMA_ACADEMICO tpa ON tpa.CODCARRERA =hd.COD_CARRERA and tpa.CODDIVISION =hd.COD_DIVISION AND tpa.CODESPECIALIZ =hd.COD_ESPECIALIZ WHERE tm.codigomateria IN ('') AND termino='2S' AND ANIO='2025' AND hd.ESTADO_MAT_TOMADA <>'AN' ;  "

In [71]:
# Fetch all rows
data_list = get_data_from_query(sql_estudiantes_viendo_pre)

In [72]:
data_list

[]

In [73]:
if len(data_list) == 0:
    print("⚠️ No se encontraron estudiantes viendo los prerrequisitos. Esto puede deberse a que no hay prerrequisitos o a un error en la consulta.")
    print("Si no ven prerequisito, quiere decir que son primera linea. No se toma en cuenta.")
    # dataframe vacio con columnas COD_ESTUDIANTE, COD_MATERIA_ACAD, NOMBRE, CARRERA, ANIO, TERMINO, VEZ_TOMADA
    df_estudiantes_viendo_pre = pd.DataFrame(columns=["COD_ESTUDIANTE", "COD_MATERIA_ACAD", "NOMBRE", "CARRERA", "ANIO", "TERMINO", "VEZ_TOMADA"])
else:
    df_estudiantes_viendo_pre = pd.DataFrame(data_list)[["COD_ESTUDIANTE", "COD_MATERIA_ACAD", "NOMBRE", "CARRERA", "ANIO", "TERMINO", "VEZ_TOMADA"]].copy()

⚠️ No se encontraron estudiantes viendo los prerrequisitos. Esto puede deberse a que no hay prerrequisitos o a un error en la consulta.
Si no ven prerequisito, quiere decir que son primera linea. No se toma en cuenta.


In [74]:
df_estudiantes_viendo_pre

,COD_ESTUDIANTE,COD_MATERIA_ACAD,NOMBRE,CARRERA,ANIO,TERMINO,VEZ_TOMADA


In [75]:
# hacer antes el copy porque si no no sabe si es en el copy o en el original
df_estudiantes_viendo_pre["COD_MATERIA_ACAD"] = df_estudiantes_viendo_pre["COD_MATERIA_ACAD"].str.strip() 

In [76]:
"Tamaño coincide", df_estudiantes_viendo_pre["COD_ESTUDIANTE"].shape[0] == len(data_list)

('Tamaño coincide', True)

In [77]:
df_estudiantes_viendo_pre = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["CARRERA"].isin(lis_carreras_pre)]

In [78]:
df_estudiantes_viendo_pre["CARRERA"].value_counts()

Series([], Name: count, dtype: int64)

In [79]:
df_estudiantes_viendo_pre["NOMBRE"].value_counts()

Series([], Name: count, dtype: int64)

In [80]:
df_estudiantes_viendo_pre

,COD_ESTUDIANTE,COD_MATERIA_ACAD,NOMBRE,CARRERA,ANIO,TERMINO,VEZ_TOMADA


In [81]:
# Estudiantes que no deberian estar nuevamente si se quedan por tercera
list_tercera = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["VEZ_TOMADA"] > 2]["COD_MATERIA_ACAD"]
list_tercera

Series([], Name: COD_MATERIA_ACAD, dtype: object)

In [82]:
lis_cod_materias_pre

[]

In [83]:
if df_estudiantes_viendo_pre.shape[0] > 0:
    dict_results = {}
    for cod_materia in lis_cod_materias_pre:
        list_matricula = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["COD_MATERIA_ACAD"] == cod_materia]["COD_ESTUDIANTE"].tolist()
        if len(list_matricula) == 0:
            print(f"⚠️ No se encontraron estudiantes viendo el prerrequisito {cod_materia}. Esto puede deberse a que no hay estudiantes actualmente viendo ese prerrequisito o a un error en la consulta.")
            continue

        results, mi_df, stats, list_student_off = make_analysis(list_matricula, cod_materia)

        if len(results) == 0:
            dificultad_materia_obj = df_complete[df_complete["COD_MATERIA_ACAD_MO"] == cod_materia_objetivo]["DIFICULTAD_MO"].unique()[0]

            list_students_unique = df_estudiantes_viendo_pre["COD_ESTUDIANTE"].unique()
            # quedarme con el ultimo registro unico de cada estudiante en df_complete, el año mayor y termino mayor
            df_estudiantes_viendo_no_analizados = df_complete[df_complete["COD_ESTUDIANTE"].isin(list_students_unique)].sort_values(["anio", "termino_num"], ascending=[False, False])
            df_estudiantes_viendo_no_analizados['CANT_ACTUAL_MAT_TOMADAS'] = df_estudiantes_viendo_no_analizados.groupby('COD_ESTUDIANTE')['COD_MATERIA_ACAD_MO'].transform('count')
            df_estudiantes_viendo_no_analizados = df_estudiantes_viendo_no_analizados.drop_duplicates(subset=['COD_ESTUDIANTE'], keep='first')
            
            df_estudiantes_viendo_no_analizados["anio"] = df_estudiantes_viendo_no_analizados["anio"].astype(int)
            df_estudiantes_viendo_no_analizados["DIFICULTAD_MO"] = dificultad_materia_obj
            df_estudiantes_viendo_no_analizados["COD_MATERIA_ACAD_MO"] = cod_materia_objetivo
            df_estudiantes_viendo_no_analizados["ESTADO_MAT_TOMADA_MO"] = "AC"
            df_con_gpa_socioeconomico_interes = complete_data_to_model(df_estudiantes_viendo_no_analizados, df_gpa_general, df_socioeconomico)
            df_con_gpa_socioeconomico_interes_clean = limpieza_df_con_gpa_socioeconomico_interes(df_con_gpa_socioeconomico_interes, label_encoders)

            results = predict_academic_risk(modelo, feature_info, df_con_gpa_socioeconomico_interes_clean, return_dataframe=True)
            stats = results['statistics']
            mi_df = df_con_gpa_socioeconomico_interes_clean.copy()
        dict_results[cod_materia] = {
            "results": results,
            "dataframe": mi_df,
            "statistics": stats
        }

In [84]:
if df_estudiantes_viendo_pre.shape[0] > 0:
    dict_results.keys()

In [85]:
if df_estudiantes_viendo_pre.shape[0] > 0:
    df_estudiantes_viendo_pre_tmp = df_estudiantes_viendo_pre.copy()
    df_estudiantes_viendo_pre_tmp["APROBADO"] = 0

In [86]:
if df_estudiantes_viendo_pre.shape[0] > 0:
    for cod_materia in dict_results.keys():
        list_matricula = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["COD_MATERIA_ACAD"] == cod_materia]["COD_ESTUDIANTE"].tolist()
        list_result = dict_results[cod_materia]["results"]["predictions"]
        if len(list_matricula) != len(list_result):
            print(f"⚠️ Desalineación: {cod_materia} - {len(list_matricula)} estudiantes, {len(list_result)} predicciones")
            print("IDs de estudiantes:", list_matricula)
            print("IDs con datos completos:", dict_results[cod_materia]["dataframe"]["COD_ESTUDIANTE"].tolist())
            # Estudiantes que faltan datos:
            ids_con_datos = set(dict_results[cod_materia]["dataframe"]["COD_ESTUDIANTE"].tolist())
            ids_sobrantes = [i for i in list_matricula if i not in ids_con_datos]
            print("Estudiantes sobrantes (sin datos completos):", ids_sobrantes)
        for i, pred in zip(list_matricula, list_result):
            df_estudiantes_viendo_pre_tmp.loc[
                (df_estudiantes_viendo_pre_tmp["COD_ESTUDIANTE"] == i) & 
                (df_estudiantes_viendo_pre_tmp["COD_MATERIA_ACAD"] == cod_materia), 
                "APROBADO"
            ] = pred

In [87]:
if df_estudiantes_viendo_pre.shape[0] > 0:
    print(df_estudiantes_viendo_pre.shape, df_estudiantes_viendo_pre_tmp.shape, df_estudiantes_viendo_pre_tmp["APROBADO"].value_counts())

In [88]:
if df_estudiantes_viendo_pre.shape[0] > 0:
    df_estudiantes_viendo_pre_tmp

### Analizar los estudiantes actuales (2025-2S) con el modelo actual, para quedarnos con los reprobados para la planificacion 2026


In [89]:
# Estudiantes viendo la materia objetivo
list_matricula = df_estudiantes_viendo["COD_ESTUDIANTE"].tolist()
len(list_matricula), df_estudiantes_viendo["COD_ESTUDIANTE"].nunique()

(0, 0)

In [90]:
df_estudiantes_viendo_tmp = df_estudiantes_viendo.copy()
df_estudiantes_viendo_tmp["APROBADO"] = 0

In [91]:
results, mi_df, stats, list_student_off = make_analysis(list_matricula, cod_materia_objetivo)

file_dir ../data/riesgo_academico/saved/inference_data.csv
✅ Datos de inferencia cargados: 30324 registros

✅ ESTT3002 encontrado!
   Valor original: ESTT3002, Valor encoded: 304
⚠️ No se encontraron datos para la materia 'ESTT3002' (encoded: 304) en los datos de inferencia.
⚠️ No se encontraron datos para los estudiantes proporcionados. Asegúrate de que las matrículas sean correctas y que los estudiantes hayan tomado la materia objetivo.


In [92]:
df_con_gpa_socioeconomico_interes_clean = pd.DataFrame()
if len(results) == 0 and df_estudiantes_viendo_tmp.shape[0] > 0:
    print("⚠️ No se encontraron resultados para los estudiantes viendo la materia objetivo. Esto puede deberse a que no hay datos completos para esos estudiantes o a un error en la consulta.")
    print("Es necesario que se completen todos los datos para que se analice correctamente el riesgo académico de los estudiantes viendo la materia objetivo.")
    need_reanalysis = True
    dificultad_materia_obj = df_complete[df_complete["COD_MATERIA_ACAD_MO"] == cod_materia_objetivo]["DIFICULTAD_MO"].unique()[0]

    list_students_unique = df_estudiantes_viendo_tmp["COD_ESTUDIANTE"].unique()
    # quedarme con el ultimo registro unico de cada estudiante en df_complete, el año mayor y termino mayor
    df_estudiantes_viendo_no_analizados = df_complete[df_complete["COD_ESTUDIANTE"].isin(list_students_unique)].sort_values(["anio", "termino_num"], ascending=[False, False])
    df_estudiantes_viendo_no_analizados['CANT_ACTUAL_MAT_TOMADAS'] = df_estudiantes_viendo_no_analizados.groupby('COD_ESTUDIANTE')['COD_MATERIA_ACAD_MO'].transform('count')
    df_estudiantes_viendo_no_analizados = df_estudiantes_viendo_no_analizados.drop_duplicates(subset=['COD_ESTUDIANTE'], keep='first')
    
    df_estudiantes_viendo_no_analizados["anio"] = df_estudiantes_viendo_no_analizados["anio"].astype(int)
    df_estudiantes_viendo_no_analizados["DIFICULTAD_MO"] = dificultad_materia_obj
    df_estudiantes_viendo_no_analizados["COD_MATERIA_ACAD_MO"] = cod_materia_objetivo
    df_estudiantes_viendo_no_analizados["ESTADO_MAT_TOMADA_MO"] = "AC"
    df_con_gpa_socioeconomico_interes = complete_data_to_model(df_estudiantes_viendo_no_analizados, df_gpa_general, df_socioeconomico)
    df_con_gpa_socioeconomico_interes_clean = limpieza_df_con_gpa_socioeconomico_interes(df_con_gpa_socioeconomico_interes, label_encoders)

    results = predict_academic_risk(modelo, feature_info, df_con_gpa_socioeconomico_interes_clean, return_dataframe=True)
    stats = results['statistics']

    print(df_con_gpa_socioeconomico_interes_clean.shape, len(results["predictions"]))
    df_con_gpa_socioeconomico_interes_clean["ESTADO_MAT_TOMADA_MO"] = df_con_gpa_socioeconomico_interes_clean.apply(
        lambda row: 'AP' if results["predictions"][row.name] == 1 else 'RP',
        axis=1
    )


In [93]:
index_matricula = 0
for i in list_matricula:
    if i in list_student_off:
        # no agregar a ningun lado
        # df_estudiantes_viendo_tmp.loc[(df_estudiantes_viendo_tmp["COD_ESTUDIANTE"] == i), "APROBADO"] = "OFF"
        continue
    # print("Procesando estudiante:", i, "Índice:", index_matricula)
    df_estudiantes_viendo_tmp.loc[(df_estudiantes_viendo_tmp["COD_ESTUDIANTE"] == i), "APROBADO"] = results["predictions"][index_matricula]
    index_matricula += 1

In [94]:
df_estudiantes_viendo.shape, df_estudiantes_viendo_tmp.shape, df_estudiantes_viendo_tmp["APROBADO"].value_counts()

((0, 7), (0, 8), Series([], Name: count, dtype: int64))

### Estudiantes que han visto el minimo numero de terminos para ver la materia objetivo que no tiene prerequisitos, y que no han visto la materia objetivo.

In [95]:
df_estudiantes_materia_sin_prerequisito = pd.DataFrame()
if df_pre_requisito.shape[0] == 0:
    estudiantes_con_materia = df_complete[df_complete['COD_MATERIA_ACAD_MO'] == cod_materia_objetivo]['COD_ESTUDIANTE']

    for row in df_pre_requisito_cp.itertuples():
        print("Carrera:", row.CARRERA, "Nivel:", row.NIVEL, "NUM_MIN_TERMINOS:", row.NUM_MIN_TERMINOS)
        df_estudiantes_materia_sin_prerequisito = pd.concat((df_estudiantes_materia_sin_prerequisito, df_complete[
            (~df_complete['COD_ESTUDIANTE'].isin(estudiantes_con_materia)) & 
            (df_complete['CARRERA'] == row.CARRERA) & 
            (df_complete["TERMINOS_REGISTRADOS"] >= row.NUM_MIN_TERMINOS)
        ]), ignore_index=True)
    
    print("Estudiantes sin prerrequisito:", df_estudiantes_materia_sin_prerequisito.shape[0])
    print("Cantidad de estudiantes unicos sin prerrequisito:", df_estudiantes_materia_sin_prerequisito["COD_ESTUDIANTE"].nunique())
    # quedarme con el ultimo registro de df_estudiantes_materia_sin_prerequisito por cada estudiante COD_ESTUDIANTE priorizando el mayor termino y luego el mayor anio
    df_estudiantes_materia_sin_prerequisito = df_estudiantes_materia_sin_prerequisito.sort_values(['anio', 'termino_num'], ascending=[False, False])
    df_estudiantes_materia_sin_prerequisito['CANT_ACTUAL_MAT_TOMADAS'] = df_estudiantes_materia_sin_prerequisito.groupby('COD_ESTUDIANTE')['COD_MATERIA_ACAD_MO'].transform('count')
    df_estudiantes_materia_sin_prerequisito = df_estudiantes_materia_sin_prerequisito.drop_duplicates(subset=['COD_ESTUDIANTE'], keep='first')
    print("Estudiantes sin prerrequisito (registro único por estudiante):", df_estudiantes_materia_sin_prerequisito.shape[0])


Carrera: Tecnología Superior en Mecatrónica Nivel: 100I NUM_MIN_TERMINOS: nan
Estudiantes sin prerrequisito: 0
Cantidad de estudiantes unicos sin prerrequisito: 0
Estudiantes sin prerrequisito (registro único por estudiante): 0


In [96]:
# df_estudiantes_materia_sin_prerequisito

### Estudiantes que ya han visto las prerequisitos pero no la objetivo y los reprobados de la materia objetivo. Y les tocaria ver la materia objetivo en 2026-1S

In [97]:
df_estudiantes_viendo_tmp

,COD_ESTUDIANTE,NOMBRE,CARRERA,COD_MATERIA_ACAD,ANIO,TERMINO,VEZ_TOMADA,APROBADO


In [98]:
lis_cod_materias_pre, lis_carreras_pre, df_pre_requisito["CARRERA"].unique().tolist(), cod_materia_objetivo

([], [], [], 'ESTT3002')

In [99]:
df_pre_requisito

,CARRERA,NIVEL,MATERIA_REQUISITO,CODIGOMATERIA,TIPO,TIPOMATERIA,MATERIA,NUM_MIN_TERMINOS


##### Descartar los RP y que estan viendo por tercera vez porque ya perdieron la carrera


In [100]:
if len(data_list) > 0:
    list_tercera_and_rp = df_estudiantes_viendo_pre_tmp[(df_estudiantes_viendo_pre_tmp["APROBADO"] == 0) & (df_estudiantes_viendo_pre_tmp["VEZ_TOMADA"] > 2)]["COD_ESTUDIANTE"].unique().tolist()
else:
    list_tercera_and_rp = []
df_estudiantes_viendo_tmp = df_estudiantes_viendo_tmp[~df_estudiantes_viendo_tmp["COD_ESTUDIANTE"].isin(list_tercera_and_rp)]

In [101]:
df_estudiantes_viendo_tmp.shape, df_estudiantes_viendo_tmp["APROBADO"].value_counts()

((0, 8), Series([], Name: count, dtype: int64))

In [102]:
df_complete["termino"].value_counts()

termino
1S    185377
2S    181523
Name: count, dtype: int64

##### Actualizar la columna ESTADO_MAT_TOMADA_MO en df_complete, de los prerequisitos actualmente viendo y los que actualmente estan viendo la materia objetivo.

In [103]:
if df_estudiantes_viendo_pre.shape[0] > 0:
    # materias de pre requisito, periodo actual, estudiantes viendo esas materias
    df_complete[
        (df_complete["COD_MATERIA_ACAD_MO"].isin(lis_cod_materias_pre)) & 
        (df_complete["anio"] == str(anio_base)) & 
        (df_complete["termino"] == str(termino_base)+"S") & 
        (df_complete["COD_ESTUDIANTE"].isin(df_estudiantes_viendo_pre_tmp["COD_ESTUDIANTE"]))
    ]

In [104]:
# Mapeo de valores de APROBADO
mapping = {1: 'AP', 0: 'RP'}

In [105]:
# Actualizar df_complete con los que vieron la prerequisito y su estado de aprobado o reprobado
if len(data_list) > 0:
    for index, row in df_estudiantes_viendo_pre_tmp.iterrows():
        # print("Procesando estudiante:", row['COD_ESTUDIANTE'], "Materia:", row['COD_MATERIA_ACAD'], "Aprobado:", row['APROBADO'])
        df_complete.loc[(df_complete['COD_ESTUDIANTE'] == row['COD_ESTUDIANTE']) & 
                        (df_complete['COD_MATERIA_ACAD_MO'] == row['COD_MATERIA_ACAD']) & 
                        (df_complete['anio'] == str(anio_base)) & 
                        (df_complete['termino'] == str(termino_base)+"S"), 
                        'ESTADO_MAT_TOMADA_MO'] = mapping[row['APROBADO']]  

In [106]:
# df_complete[(df_complete["COD_ESTUDIANTE"] == '202515714') & (df_complete["COD_MATERIA_ACAD_MO"] == 'CCPG1043')]

In [107]:
# Actualizar df_complete con los que vieron la materia objetivo y su estado de aprobado o reprobado
for index, row in df_estudiantes_viendo_tmp.iterrows():
    # print("Procesando estudiante:", row['COD_ESTUDIANTE'], "Materia:", row['COD_MATERIA_ACAD'], "Aprobado:", row['APROBADO'])
    df_complete.loc[(df_complete['COD_ESTUDIANTE'] == row['COD_ESTUDIANTE']) & 
                    (df_complete['COD_MATERIA_ACAD_MO'] == row['COD_MATERIA_ACAD']) & 
                    (df_complete['anio'] == str(anio_base)) & 
                    (df_complete['termino'] == str(termino_base)+"S"), 
                    'ESTADO_MAT_TOMADA_MO'] = mapping[row['APROBADO']]  

In [108]:
# df_complete[(df_complete["COD_ESTUDIANTE"] == '202400701') & (df_complete["COD_MATERIA_ACAD_MO"] == 'MATG1058')]

### Obtener los estudiantes que veran la materia objetivo en 2026-1S

#### 1. Repitentes: Estudiantes con último estado RP o PF en la materia objetivo

In [109]:
# Filtrar estudiantes que han cursado la materia objetivo
df_cursaron_objetivo = df_complete[df_complete['COD_MATERIA_ACAD_MO'] == cod_materia_objetivo].copy()

# Verificar si alguna vez aprobaron (AP) la materia objetivo
estudiantes_con_ap = df_cursaron_objetivo[df_cursaron_objetivo['ESTADO_MAT_TOMADA_MO'] == 'AP']['COD_ESTUDIANTE'].unique()

# Filtrar solo estudiantes que nunca aprobaron (excluir los que tienen AP)
df_sin_ap = df_cursaron_objetivo[~df_cursaron_objetivo['COD_ESTUDIANTE'].isin(estudiantes_con_ap)]

print("Cantidad de estudiantes que cursaron la materia objetivo mas de dos vez sin aprobar:", df_sin_ap['VEZ_TOMADA_MO'].value_counts()[2:].sum())

# Filtrar solo RP o PF y han visto la materia mas de dos veces VEZ_TOMADA_MO
df_rp_pf = df_sin_ap[(df_sin_ap['ESTADO_MAT_TOMADA_MO'].isin(['RP', 'PF'])) & (df_sin_ap['VEZ_TOMADA_MO'] < 3)].copy()
print("Deberian ser igual: ", df_sin_ap.shape[0] == df_rp_pf.shape[0])

# Ordenar por estudiante y fecha para obtener el último registro
df_rp_pf = df_rp_pf.sort_values(['anio', 'termino_num'], ascending=[True, True])

# Obtener el último registro por estudiante
df_repitentes = df_rp_pf.groupby('COD_ESTUDIANTE').first().reset_index()

Cantidad de estudiantes que cursaron la materia objetivo mas de dos vez sin aprobar: 0
Deberian ser igual:  True


In [110]:
df_repitentes.shape

(0, 32)

In [111]:
print(f"Total de repitentes: {df_repitentes.shape[0]}")
df_repitentes.head()

Total de repitentes: 0


,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num


In [112]:
df_repitentes.groupby(["anio", "termino"]).size()

Series([], dtype: int64)

In [113]:
df_repitentes["ESTADO_MAT_TOMADA_MO"].value_counts()

Series([], Name: count, dtype: int64)

#### 2. Nuevos: Estudiantes que cumplen prerrequisitos y nunca cursaron la materia objetivo

In [114]:
# 1. Obtener las materias prerrequisito por carrera
prereq_por_carrera = (
    df_pre_requisito.groupby('CARRERA')['CODIGOMATERIA']
    .apply(lambda x: set(x.str.strip()))
    .reset_index()
    .rename(columns={'CODIGOMATERIA': 'materias_requeridas'})
)

In [115]:
# 2. Filtrar registros de df_complete donde el estudiante APROBÓ los prerrequisitos
df_complete_prereq_aprobados = df_complete[
    (df_complete['COD_MATERIA_ACAD_MO'].isin(lis_cod_materias_pre)) &
    (df_complete['CARRERA'].isin(lis_carreras_pre)) &
    (df_complete['ESTADO_MAT_TOMADA_MO'] == 'AP')  # Solo materias aprobadas
].copy()

In [116]:
# 3. Agrupar por estudiante y carrera para obtener materias aprobadas
materias_aprobadas_por_estudiante = (
    df_complete_prereq_aprobados.groupby(['COD_ESTUDIANTE', 'CARRERA'])['COD_MATERIA_ACAD_MO']
    .apply(lambda x: set(x.str.strip() if hasattr(x, 'str') else x))
    .reset_index()
    .rename(columns={'COD_MATERIA_ACAD_MO': 'materias_aprobadas'})
)

In [117]:
materias_aprobadas_por_estudiante


,COD_ESTUDIANTE,CARRERA,materias_aprobadas


In [118]:
prereq_por_carrera

,CARRERA,materias_requeridas


In [119]:
# 4. Hacer merge para comparar con prerrequisitos requeridos
df_comparacion = pd.merge(
    materias_aprobadas_por_estudiante,
    prereq_por_carrera,
    on='CARRERA',
    how='inner'
)

In [120]:
# 5. Verificar que tengan TODOS los prerrequisitos aprobados
if not df_comparacion.empty:
    df_comparacion['tiene_todos_prereq'] = df_comparacion.apply(
        lambda row: row['materias_requeridas'].issubset(row['materias_aprobadas']),
        axis=1
    )
    estudiantes_con_prereq_completos = df_comparacion[df_comparacion['tiene_todos_prereq']]
else:
    estudiantes_con_prereq_completos = pd.DataFrame()

In [121]:
# 6. Filtrar estudiantes que NUNCA han visto la materia objetivo
estudiantes_vieron_objetivo = set(
    df_complete[df_complete['COD_MATERIA_ACAD_MO'] == cod_materia_objetivo]['COD_ESTUDIANTE']
)

In [122]:
# 7. Resultado final: estudiantes elegibles (con prereq aprobados y sin haber visto objetivo)
if not estudiantes_con_prereq_completos.empty and 'COD_ESTUDIANTE' in estudiantes_con_prereq_completos.columns:
    df_nuevos = estudiantes_con_prereq_completos[
        ~estudiantes_con_prereq_completos['COD_ESTUDIANTE'].isin(estudiantes_vieron_objetivo)
    ].copy()
else:
    df_nuevos = pd.DataFrame()

In [123]:
if len(estudiantes_con_prereq_completos) > 0:
    print(f"📊 Resumen:")
    print(f"  - Estudiantes con todos los prerrequisitos APROBADOS: {len(estudiantes_con_prereq_completos)}")
    print(f"  - Estudiantes que vieron {cod_materia_objetivo}: {len(estudiantes_vieron_objetivo)}")
    print(f"  - Estudiantes NUEVOS elegibles: {len(df_nuevos)}")
    print(f"\nDistribución por carrera:")
    print(df_nuevos['CARRERA'].value_counts())
    df_nuevos[['COD_ESTUDIANTE', 'CARRERA', 'materias_aprobadas', 'materias_requeridas']]    

In [124]:
if len(estudiantes_con_prereq_completos) > 0 and not df_nuevos.empty:
    # 1. Validación final: Verificar que TODOS los prerrequisitos estén cumplidos
    print("🔍 Validación de prerrequisitos por estudiante:")
    for idx, row in df_nuevos.iterrows():
        faltantes = row['materias_requeridas'] - row['materias_aprobadas']
        if len(faltantes) > 0:
            print(f"  ⚠️ Estudiante {row['COD_ESTUDIANTE']} - Carrera: {row['CARRERA']} - Faltantes: {faltantes}")

    print(f"\n✅ Validación completa. Total estudiantes con prerrequisitos completos: {len(df_nuevos)}")
    # Verificar que no haya faltantes
    df_nuevos['prereq_completos'] = df_nuevos.apply(
        lambda row: len(row['materias_requeridas'] - row['materias_aprobadas']) == 0,
        axis=1
    )
    print(f"\n✅ Todos cumplen prerrequisitos: {df_nuevos['prereq_completos'].all()}")
    print(f"Total estudiantes con prerrequisitos completos: {df_nuevos['prereq_completos'].sum()}")

In [125]:
df_nuevos

""


In [126]:
# df_estudiantes_materia_sin_prerequisito

In [127]:
# cod_materia_objetivo

In [128]:
# para comprobar que ese estudiante no ha visto la materia objetivo y si ha visto las materias requestidas
# df_complete[(df_complete['COD_ESTUDIANTE'] == "201229676") & (df_complete['COD_MATERIA_ACAD_MO'] == "MATG1057")]

#### 3. Unión: DataFrame final con Repitentes y Nuevos

In [129]:
# Agregar columna identificadora del tipo de estudiante
df_repitentes['TIPO_ESTUDIANTE'] = 'REPITENTE'
if len(df_nuevos) > 0:
    df_nuevos['TIPO_ESTUDIANTE'] = 'NUEVO'
    # Unir ambos dataframes
    df_final_nuevos_y_repitentes = pd.concat([df_repitentes, df_nuevos], ignore_index=True)
elif df_estudiantes_materia_sin_prerequisito.shape[0] > 0:
    df_estudiantes_materia_sin_prerequisito['TIPO_ESTUDIANTE'] = 'SIN_PREREQUISITO'
    # Unir ambos dataframes
    df_final_nuevos_y_repitentes = pd.concat([df_repitentes, df_estudiantes_materia_sin_prerequisito], ignore_index=True)
else:
    df_final_nuevos_y_repitentes = df_repitentes.copy()


# Ordenar por tipo y código de estudiante
df_final_nuevos_y_repitentes = df_final_nuevos_y_repitentes.sort_values(['TIPO_ESTUDIANTE', 'COD_ESTUDIANTE']).reset_index(drop=True)

print(f"\n📊 RESUMEN FINAL:")
print(f"  • Repitentes: {df_repitentes.shape[0]}")
print(f"  • Nuevos: {df_nuevos.shape[0]}")
print(f"  • Estudiantes de materia Sin prerrequisito: {df_estudiantes_materia_sin_prerequisito.shape[0]}")
print(f"  • TOTAL: {df_final_nuevos_y_repitentes.shape[0]}")
print(f"\nDistribución por tipo:")
print(df_final_nuevos_y_repitentes['TIPO_ESTUDIANTE'].value_counts())

df_final_nuevos_y_repitentes


📊 RESUMEN FINAL:
  • Repitentes: 0
  • Nuevos: 0
  • Estudiantes de materia Sin prerrequisito: 0
  • TOTAL: 0

Distribución por tipo:
Series([], Name: count, dtype: int64)


,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,TIPO_ESTUDIANTE


In [130]:
# Verificación: mostrar ejemplos de cada tipo
print("\n🔍 Ejemplos de REPITENTES:")
print(df_final_nuevos_y_repitentes[df_final_nuevos_y_repitentes['TIPO_ESTUDIANTE'] == 'REPITENTE'][['COD_ESTUDIANTE', 'TIPO_ESTUDIANTE', 'ESTADO_MAT_TOMADA_MO', 'anio', 'termino']].head())

print("\n🔍 Ejemplos de NUEVOS:")
print(df_final_nuevos_y_repitentes[df_final_nuevos_y_repitentes['TIPO_ESTUDIANTE'] == 'NUEVO'][['COD_ESTUDIANTE', 'TIPO_ESTUDIANTE']].head())

print("\n🔍 Ejemplos de SIN PREREQUISITO:")
print(df_final_nuevos_y_repitentes[df_final_nuevos_y_repitentes['TIPO_ESTUDIANTE'] == 'SIN_PREREQUISITO'][['COD_ESTUDIANTE', 'TIPO_ESTUDIANTE']].head())


🔍 Ejemplos de REPITENTES:
Empty DataFrame
Columns: [COD_ESTUDIANTE, TIPO_ESTUDIANTE, ESTADO_MAT_TOMADA_MO, anio, termino]
Index: []

🔍 Ejemplos de NUEVOS:
Empty DataFrame
Columns: [COD_ESTUDIANTE, TIPO_ESTUDIANTE]
Index: []

🔍 Ejemplos de SIN PREREQUISITO:
Empty DataFrame
Columns: [COD_ESTUDIANTE, TIPO_ESTUDIANTE]
Index: []


In [131]:
df_final_nuevos_y_repitentes

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,TIPO_ESTUDIANTE


In [132]:
df_final_nuevos_y_repitentes.keys() # mejor solo agregar la dificultad de la materia objetivo

Index(['COD_ESTUDIANTE', 'COD_MATERIA_ACAD_MO', 'ESTADO_MAT_TOMADA_MO',
       'VEZ_TOMADA_MO', 'NOTA1_MO', 'NOTA2MO', 'PROMEDIO_MO', 'DIFICULTAD_MO',
       'T_MAT_TOMADAS', 'PROM_1PARCIAL', 'PROM_2PARCIAL',
       'PROM_CALIFICACIONES', 'MAT_APROBADAS', 'PROM_CALIF_APROBADAS',
       'TERMINOS_REGISTRADOS', 'PERDIO_CARRERA', 'PROM_MAT_REPROBADAS1',
       'PROM_MAT_REPROBADAS2', 'PROM_MAT_REPROBADAS3', 'MUY_FACIL', 'FACIL',
       'MODERADA', 'DIFICIL', 'MUY_DIFICIL', 'promedio_general',
       'CODIGOMATERIA', 'MATERIA', 'anio', 'termino', 'PROMEDIO_GENERAL',
       'CARRERA', 'termino_num', 'TIPO_ESTUDIANTE'],
      dtype='object')

#### Obtener los ultimos registros de los estudiantes que veran la materia objetivo (df_final_nuevos_y_repitentes["COD_ESTUDIANTE"].unique())

In [133]:
df_final_nuevos_y_repitentes["COD_ESTUDIANTE"].nunique(), df_final_nuevos_y_repitentes.shape[0]

(0, 0)

In [134]:
# 1. Filtrar el df_complete para tener solo los estudiantes de df_final_nuevos_y_repitentes
estudiantes_ids = df_final_nuevos_y_repitentes["COD_ESTUDIANTE"].unique()
df_filtrado = df_complete[df_complete["COD_ESTUDIANTE"].isin(estudiantes_ids)].copy()
# eliminar los que tienen una VEZ_TOMADA_MO = 3 y ESTADO_MAT_TOMADA_MO = RP o PF
df_filtrado = df_filtrado[~((df_filtrado["VEZ_TOMADA_MO"] == 3) & (df_filtrado["ESTADO_MAT_TOMADA_MO"].isin(["RP", "PF"])))]

In [135]:
# 2. Crear un identificador numérico único para el periodo (Año + Término)
# Multiplicamos el año por 100 para que 2020-2 sea mayor que 2020-1 de forma matemática
df_filtrado['periodo_id'] = (df_filtrado['anio'].astype(int) * 100) + df_filtrado['termino_num']

In [136]:
# 3. Calcular el periodo MÁXIMO por cada estudiante y asignarlo a una nueva columna
# transform('max') repite el valor máximo del grupo en todas las filas de ese estudiante
df_filtrado['max_periodo'] = df_filtrado.groupby('COD_ESTUDIANTE')['periodo_id'].transform('max')

In [137]:
# 4. Filtrar las filas donde el periodo actual coincide con el máximo encontrado
df_resultado = df_filtrado[df_filtrado['periodo_id'] == df_filtrado['max_periodo']].copy()

In [138]:
# Limpiar columnas auxiliares
df_resultado.drop(columns=['periodo_id', 'max_periodo'], inplace=True)

In [139]:
df_resultado

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num


In [140]:
list_student_resultado = df_resultado["COD_ESTUDIANTE"].unique()

In [141]:
len(list_student_resultado), len(estudiantes_ids)

(0, 0)

In [142]:
indice_busq_para_comprobar = 0

In [143]:
list_student_resultado

array([], dtype=object)

In [144]:
# df_resultado[df_resultado["COD_ESTUDIANTE"] == list_student_resultado[indice_busq_para_comprobar]]

In [145]:
if len(list_student_resultado) > 0:
    df_complete[df_complete["COD_ESTUDIANTE"] == list_student_resultado[indice_busq_para_comprobar]].sort_values(by=['anio', 'termino_num'], ascending=[False, False])

In [146]:
df_resultado

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num


#### Quedarme con el mayor y actualiza su DIFICULTAD_MO

In [147]:
dificultad_materia_obj = df_complete[df_complete["COD_MATERIA_ACAD_MO"] == cod_materia_objetivo]["DIFICULTAD_MO"].unique()[0]

In [148]:
# 1. Calcular la diferencia absoluta con la dificultad objetivo
# (Asumiendo que 'dificultad_materia_obj' es una variable con el valor numérico)
df_resultado['diff_gap'] = (df_resultado['DIFICULTAD_MO'].astype(float) - dificultad_materia_obj).abs()

In [149]:
# 2. Ordenar los datos
# - Primero por estudiante (para agrupar)
# - Segundo por 'diff_gap' ASCENDENTE (el más cercano a 0 es el más similar)
# - Tercero por 'PROMEDIO_MO' DESCENDENTE (el más alto gana en caso de empate o cercanía similar)
df_ordenado = df_resultado.sort_values(
    by=['COD_ESTUDIANTE', 'diff_gap', 'PROMEDIO_MO'],
    ascending=[True, True, False]
)

In [150]:
# agregar la columna CANT_ACTUAL_MAT_TOMADAS  a df_ordenado antes de eliminar los duplicados
# df_ordenado['CANT_ACTUAL_MAT_TOMADAS'] = df_ordenado.groupby('COD_ESTUDIANTE').cumcount() + 1
df_ordenado['CANT_ACTUAL_MAT_TOMADAS'] = df_ordenado.groupby('COD_ESTUDIANTE')['COD_MATERIA_ACAD_MO'].transform('count')

In [151]:
# 3. Quedarse con la primera fila de cada estudiante (la mejor opción según el orden)
df_seleccion_final = df_ordenado.drop_duplicates(subset='COD_ESTUDIANTE', keep='first').copy()

In [152]:
# Limpieza opcional
df_seleccion_final.drop(columns=['diff_gap'], inplace=True)

In [153]:
df_seleccion_final

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,CANT_ACTUAL_MAT_TOMADAS


In [154]:
df_seleccion_final["COD_ESTUDIANTE"].nunique(), df_final_nuevos_y_repitentes["COD_ESTUDIANTE"].nunique(), df_resultado["COD_ESTUDIANTE"].nunique()

(0, 0, 0)

In [155]:
df_seleccion_final["DIFICULTAD_MO"] = dificultad_materia_obj
df_seleccion_final["COD_MATERIA_ACAD_MO"] = cod_materia_objetivo
df_seleccion_final["ESTADO_MAT_TOMADA_MO"] = "AC"

### Usar el modelo para 2026-1S

In [156]:
df_seleccion_final["anio"] = df_seleccion_final["anio"].astype(int)

In [157]:
df_seleccion_final

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,CANT_ACTUAL_MAT_TOMADAS


In [158]:
df_final_nuevos_y_repitentes

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,TIPO_ESTUDIANTE


In [159]:
df_final_nuevos_y_repitentes[df_final_nuevos_y_repitentes["TIPO_ESTUDIANTE"] == "NUEVO"]["COD_ESTUDIANTE"].nunique()

0

In [160]:
df_seleccion_final

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,CANT_ACTUAL_MAT_TOMADAS


In [161]:
df_con_gpa_socioeconomico_interes = complete_data_to_model(df_seleccion_final, df_gpa_general, df_socioeconomico)

c:\Users\saraujo\Documents\Riesgo academico\Codigos_riesgo_academico\planificacion_aprobacion\complete_data_to_model.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_con_gpa_socioeconomico["GASTOS_RUBRO"].fillna(df_con_gpa_socioeconomico["GASTOS_RUBRO"].mean(), inplace=True)
c:\Users\saraujo\Documents\Riesgo academico\Codigos_riesgo_academico\planificacion_aprobacion\complete_data_to_model.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_in

In [162]:
df_con_gpa_socioeconomico_interes_clean = limpieza_df_con_gpa_socioeconomico_interes(df_con_gpa_socioeconomico_interes, label_encoders)

🔍 Verificando valores en cada columna categórica...



In [163]:
df_con_gpa_socioeconomico_interes_clean

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,NIVELINSTRUCCIONPADRE_encoded,NIVELINSTRUCCIONMADRE_encoded,ESTADOCIVILPADRES_encoded,FAMILIARDISCAPACIDAD_encoded,FAMILIARENFERMEDAD_encoded,TIPOPARROQUIA_encoded,VIVEGRUPOFAMILIAR_encoded,SEXO_encoded,PERDIO_CARRERA_encoded,termino_encoded


In [164]:
# feature_info

In [165]:
df_con_gpa_socioeconomico_interes_clean

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,NIVELINSTRUCCIONPADRE_encoded,NIVELINSTRUCCIONMADRE_encoded,ESTADOCIVILPADRES_encoded,FAMILIARDISCAPACIDAD_encoded,FAMILIARENFERMEDAD_encoded,TIPOPARROQUIA_encoded,VIVEGRUPOFAMILIAR_encoded,SEXO_encoded,PERDIO_CARRERA_encoded,termino_encoded


In [166]:
results = predict_academic_risk(modelo, feature_info, df_con_gpa_socioeconomico_interes_clean, return_dataframe=True)
stats = results['statistics']

📊 Usando DataFrame proporcionado


ValueError: No hay datos disponibles para inferencia

In [ ]:
# results

In [ ]:
# stats

In [ ]:
df_con_gpa_socioeconomico_interes_clean.shape, len(results["predictions"])

((158, 85), 158)

### Save dataframe with predictions

In [ ]:
# Para la primera ejecucion va comentado todo 
df_new_semester = pd.read_csv("../data/riesgo_academico/all_2026_1S.csv")
df_seleccion_final_3 = pd.read_csv("../data/riesgo_academico/PAO_2026_1S.csv")

In [ ]:
# df_new_semester.keys()
list_keys = ['COD_ESTUDIANTE', 'COD_MATERIA_ACAD_MO',
       'ESTADO_MAT_TOMADA_MO', 'VEZ_TOMADA_MO', 'NOTA1_MO', 'NOTA2MO',
       'PROMEDIO_MO', 'DIFICULTAD_MO', 'T_MAT_TOMADAS', 'PROM_1PARCIAL',
       'PROM_2PARCIAL', 'PROM_CALIFICACIONES', 'MAT_APROBADAS',
       'PROM_CALIF_APROBADAS', 'TERMINOS_REGISTRADOS', 'PERDIO_CARRERA',
       'PROM_MAT_REPROBADAS1', 'PROM_MAT_REPROBADAS2', 'PROM_MAT_REPROBADAS3',
       'MUY_FACIL', 'FACIL', 'MODERADA', 'DIFICIL', 'MUY_DIFICIL',
       'promedio_general', 'CODIGOMATERIA', 'MATERIA']

In [ ]:
df_con_gpa_socioeconomico_interes_clean[list_keys]

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,PROM_MAT_REPROBADAS2,PROM_MAT_REPROBADAS3,MUY_FACIL,FACIL,MODERADA,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA
0,200319358,ESTG1057,AC,2,62,45,6.74,6.96,6.0,20.0,...,0.00,NaN,0,0,0,0,1,NaN,MATG1061,METAHEURÍSTICAS
1,200606465,ESTG1057,AC,2,48,26,4.96,6.96,18.0,57.0,...,NaN,NaN,0,0,0,0,2,NaN,ESTG1057,SIMULACIÓN MATEMÁTICA
2,200712586,ESTG1057,AC,1,82,51,6.45,6.96,4.0,60.0,...,NaN,NaN,0,0,0,0,1,NaN,MATG1058,OPTIMIZACIÓN NUMÉRICA
3,200828259,ESTG1057,AC,2,50,91,7.93,6.96,40.0,46.0,...,1.67,NaN,0,0,0,0,2,NaN,MATG1061,METAHEURÍSTICAS
4,200901387,ESTG1057,AC,1,0,0,0.00,6.96,45.0,56.0,...,5.45,NaN,0,1,0,0,1,NaN,CCPG1058,SISTEMAS DE INFORMACIÓN APLICADOS A LOGÍSTICA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,202412086,ESTG1057,AC,1,0,0,0.00,6.96,9.0,77.0,...,NaN,NaN,0,0,0,0,4,NaN,IDIG2012,COMUNICACIÓN
154,202412102,ESTG1057,AC,1,0,0,0.00,6.96,9.0,73.0,...,NaN,NaN,0,0,0,0,5,NaN,IDIG2012,COMUNICACIÓN
155,202413944,ESTG1057,AC,2,0,0,0.00,6.96,10.0,51.0,...,4.72,NaN,0,0,0,0,3,NaN,MATG1046,CÁLCULO VECTORIAL
156,202414652,ESTG1057,AC,1,0,0,0.00,6.96,8.0,73.0,...,NaN,NaN,0,0,0,0,3,NaN,IDIG2012,COMUNICACIÓN


In [ ]:
# df_con_gpa_socioeconomico_interes_clean["ESTADO_MAT_TOMADA_MO"] = if results["predictions"] = 0 "RP" else "AP"
df_con_gpa_socioeconomico_interes_clean["ESTADO_MAT_TOMADA_MO"] = df_con_gpa_socioeconomico_interes_clean.apply(
    lambda row: 'AP' if results["predictions"][row.name] == 1 else 'RP',
    axis=1
)

In [ ]:
df_con_gpa_socioeconomico_interes_clean

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,NIVELINSTRUCCIONPADRE_encoded,NIVELINSTRUCCIONMADRE_encoded,ESTADOCIVILPADRES_encoded,FAMILIARDISCAPACIDAD_encoded,FAMILIARENFERMEDAD_encoded,TIPOPARROQUIA_encoded,VIVEGRUPOFAMILIAR_encoded,SEXO_encoded,PERDIO_CARRERA_encoded,termino_encoded
0,200319358,ESTG1057,RP,2,62,45,6.74,6.96,6.0,20.0,...,0,0,0,297,173,1,1,1,0,0
1,200606465,ESTG1057,RP,2,48,26,4.96,6.96,18.0,57.0,...,11,12,0,297,173,1,1,1,0,1
2,200712586,ESTG1057,AP,1,82,51,6.45,6.96,4.0,60.0,...,0,0,5,297,173,1,1,1,0,0
3,200828259,ESTG1057,RP,2,50,91,7.93,6.96,40.0,46.0,...,11,11,5,191,173,1,1,1,0,1
4,200901387,ESTG1057,AP,1,0,0,0.00,6.96,45.0,56.0,...,0,0,5,297,173,1,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,202412086,ESTG1057,AP,1,0,0,0.00,6.96,9.0,77.0,...,6,6,0,297,173,1,1,0,0,1
154,202412102,ESTG1057,AP,1,0,0,0.00,6.96,9.0,73.0,...,6,6,0,297,173,1,1,0,0,1
155,202413944,ESTG1057,RP,2,0,0,0.00,6.96,10.0,51.0,...,12,11,5,297,173,1,1,1,0,1
156,202414652,ESTG1057,AP,1,0,0,0.00,6.96,8.0,73.0,...,1,4,6,297,173,1,1,0,0,1


In [ ]:
print("Para la materia objetivo:", cod_materia_objetivo, ":", materia_objetivo)

Para la materia objetivo: ESTG1057 : SIMULACIÓN MATEMÁTICA


In [ ]:
df_con_gpa_socioeconomico_interes_clean["ESTADO_MAT_TOMADA_MO"].value_counts()

ESTADO_MAT_TOMADA_MO
AP    126
RP     32
Name: count, dtype: int64

In [ ]:
# agregar al final de df_new_semester lo que corresponda de df_con_gpa_socioeconomico_interes_clean
df_tmp_new_semester = pd.concat([df_new_semester, df_con_gpa_socioeconomico_interes_clean[df_new_semester.keys()]], ignore_index=True)
# Para primera ejecucion solo
# df_tmp_new_semester = df_con_gpa_socioeconomico_interes_clean[list_keys].copy()

In [ ]:
df_tmp_new_semester

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,PROM_MAT_REPROBADAS2,PROM_MAT_REPROBADAS3,MUY_FACIL,FACIL,MODERADA,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA
0,201413291,ACUG1035,AP,1,98,95,9.48,7.90,59.0,62.0,...,4.17,NaN,1,0,2,0,0,NaN,ACUG1048,SISTEMAS ACUÍCOLAS
1,201413291,ACUG1035,AP,1,98,95,9.48,7.90,59.0,62.0,...,4.17,NaN,1,0,2,0,0,NaN,ACUG1048,SISTEMAS ACUÍCOLAS
2,201413828,ACUG1035,AP,1,81,70,7.68,7.90,48.0,62.0,...,NaN,NaN,0,0,1,0,1,NaN,ACUG1048,SISTEMAS ACUÍCOLAS
3,201416022,ACUG1035,AP,1,70,75,7.17,7.90,59.0,58.0,...,4.52,NaN,1,0,1,0,0,NaN,ACUG1045,PRODUCCIÓN ACUÍCOLA II
4,201416022,ACUG1035,AP,1,70,75,7.17,7.90,59.0,58.0,...,4.52,NaN,1,0,1,0,0,NaN,ACUG1045,PRODUCCIÓN ACUÍCOLA II
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81514,202412086,ESTG1057,AP,1,0,0,0.00,6.96,9.0,77.0,...,NaN,NaN,0,0,0,0,4,NaN,IDIG2012,COMUNICACIÓN
81515,202412102,ESTG1057,AP,1,0,0,0.00,6.96,9.0,73.0,...,NaN,NaN,0,0,0,0,5,NaN,IDIG2012,COMUNICACIÓN
81516,202413944,ESTG1057,RP,2,0,0,0.00,6.96,10.0,51.0,...,4.72,NaN,0,0,0,0,3,NaN,MATG1046,CÁLCULO VECTORIAL
81517,202414652,ESTG1057,AP,1,0,0,0.00,6.96,8.0,73.0,...,NaN,NaN,0,0,0,0,3,NaN,IDIG2012,COMUNICACIÓN


In [ ]:
df_tmp_new_semester.to_csv("../data/riesgo_academico/all_2026_1S.csv", index=False, float_format="%.2f")

In [ ]:
df_tmp_seleccion_final_3  = pd.concat([df_seleccion_final_3, df_con_gpa_socioeconomico_interes_clean], ignore_index=True)
# Para primera ejecucion solo
# df_tmp_seleccion_final_3 = df_con_gpa_socioeconomico_interes_clean.copy()

In [ ]:
df_tmp_seleccion_final_3

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,NIVELINSTRUCCIONPADRE_encoded,NIVELINSTRUCCIONMADRE_encoded,ESTADOCIVILPADRES_encoded,FAMILIARDISCAPACIDAD_encoded,FAMILIARENFERMEDAD_encoded,TIPOPARROQUIA_encoded,VIVEGRUPOFAMILIAR_encoded,SEXO_encoded,PERDIO_CARRERA_encoded,termino_encoded
0,201413291,ACUG1035,AP,1,98,95,9.48,7.90,59.0,62.0,...,5,4,0,297,173,1,1,1,0,1
1,201413291,ACUG1035,AP,1,98,95,9.48,7.90,59.0,62.0,...,5,4,0,297,173,1,1,1,0,1
2,201413828,ACUG1035,AP,1,81,70,7.68,7.90,48.0,62.0,...,4,7,0,297,173,0,1,0,1,0
3,201416022,ACUG1035,AP,1,70,75,7.17,7.90,59.0,58.0,...,12,6,0,297,173,1,1,1,0,1
4,201416022,ACUG1035,AP,1,70,75,7.17,7.90,59.0,58.0,...,12,6,0,297,173,1,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81514,202412086,ESTG1057,AP,1,0,0,0.00,6.96,9.0,77.0,...,6,6,0,297,173,1,1,0,0,1
81515,202412102,ESTG1057,AP,1,0,0,0.00,6.96,9.0,73.0,...,6,6,0,297,173,1,1,0,0,1
81516,202413944,ESTG1057,RP,2,0,0,0.00,6.96,10.0,51.0,...,12,11,5,297,173,1,1,1,0,1
81517,202414652,ESTG1057,AP,1,0,0,0.00,6.96,8.0,73.0,...,1,4,6,297,173,1,1,0,0,1


In [ ]:
df_tmp_seleccion_final_3.to_csv("../data/riesgo_academico/PAO_2026_1S.csv", index=False, float_format="%.2f")

In [ ]:
ibm_db.close(conn)

True